# Resistor YOLOX full training on Colab GPU

Runs the exact private `resistor_scanner` full pipeline: YOLOX-Tiny + YOLOX-S, 80 epochs, TFLite export, evaluation, deterministic selection, encryption/finalization.

Training uses all three public COCO datasets from `Persie0/resistor_model` release `r1`: `rres.v4i.coco.zip`, `resistor.value.training.v8i.coco.zip`, and `Deteksi.Nilai.Resistor.v1i.coco.zip`. The pipeline SHA-verifies them and builds one leakage-safe grouped 70/15/15 split.

Before **Run all**: choose a GPU runtime, then add a Colab Secret named `PRIVATE_REPO_TOKEN` (or `GH_TOKEN`) with read/write Contents access to `Persie0/resistor_scanner` and enable notebook access. The token is only needed for the private repo clone/push; the datasets are public.

In [ ]:
from google.colab import userdata
import os, pathlib, shutil, subprocess

BRANCH = 'agent/yolox-resistor-model'
repo = pathlib.Path('/content/resistor_scanner')
subprocess.run(['nvidia-smi'], check=True)

token = None
for name in ('PRIVATE_REPO_TOKEN', 'GH_TOKEN'):
    try:
        token = userdata.get(name)
    except Exception:
        token = None
    if token:
        break
if not token:
    raise RuntimeError('Add PRIVATE_REPO_TOKEN or GH_TOKEN in Colab Secrets and enable notebook access')

env = os.environ.copy()
env['GH_TOKEN'] = token
helper = '!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f'
shutil.rmtree(repo, ignore_errors=True)
subprocess.run([
    'git', '-c', f'credential.helper={helper}', 'clone',
    '--branch', BRANCH, '--single-branch',
    'https://github.com/Persie0/resistor_scanner.git', str(repo),
], check=True, env=env)
start_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
pathlib.Path('/content/resistor_start_sha.txt').write_text(start_sha)
print('Checked out', BRANCH, start_sha)
del token
env.pop('GH_TOKEN', None)

In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/resistor_scanner')
requirements = repo / 'ml/resistor_model/requirements-training.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable after setup. Select Runtime > Change runtime type > GPU, reconnect, and Run all.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, pathlib, subprocess
repo = pathlib.Path('/content/resistor_scanner')
run_env = os.environ.copy()
run_env['RUNNER_TEMP'] = '/content'
run_env['GITHUB_RUN_ID'] = 'colab'
subprocess.run([
    'bash', str(repo / '.github/playground/resistor-yolox-model.sh'), str(repo), 'full'
], check=True, cwd=repo, env=run_env)

In [ ]:
import json, pathlib, shutil
repo = pathlib.Path('/content/resistor_scanner')
artifacts = repo / 'ml/resistor_model/artifacts'
manifest = json.loads((artifacts / 'pipeline.json').read_text())
print('Datasets:')
for asset in manifest.get('dataset_assets', []):
    print('-', asset['name'], asset['sha256'])
print(json.dumps(manifest['selection'], indent=2))
zip_path = shutil.make_archive('/content/resistor-yolox-full-result', 'zip', artifacts)
print('Result bundle:', zip_path)

In [ ]:
# Safe push-back. No force push: if Actions/another process changed the branch, this stops.
from google.colab import userdata
import base64, pathlib, subprocess
BRANCH = 'agent/yolox-resistor-model'
repo = pathlib.Path('/content/resistor_scanner')
start_sha = pathlib.Path('/content/resistor_start_sha.txt').read_text().strip()
token = None
for name in ('PRIVATE_REPO_TOKEN', 'GH_TOKEN'):
    try:
        token = userdata.get(name)
    except Exception:
        token = None
    if token:
        break
if not token:
    raise RuntimeError('PRIVATE_REPO_TOKEN or GH_TOKEN is required for push-back')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_auth = ['git', '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}']
subprocess.run(git_auth + ['-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
remote_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
if remote_sha != start_sha:
    raise RuntimeError(f'Remote branch changed while training ({start_sha[:8]} -> {remote_sha[:8]}). Nothing was force-pushed.')
subprocess.run(['git', '-C', str(repo), 'config', 'user.name', 'colab-resistor-model'], check=True)
subprocess.run(['git', '-C', str(repo), 'config', 'user.email', '76871553+Persie0@users.noreply.github.com'], check=True)
paths = [
    'docs/model_reports', 'pubspec.yaml', 'lib/provider/settings_provider.dart',
    'test/services/model_asset_integrity_test.dart', 'test/provider/settings_model_test.dart',
]
if (repo / 'assets/model_yolox_v1.tflite.enc').exists():
    paths.append('assets/model_yolox_v1.tflite.enc')
subprocess.run(['git', '-C', str(repo), 'add', '--', *paths], check=True)
if subprocess.run(['git', '-C', str(repo), 'diff', '--cached', '--quiet']).returncode == 0:
    print('No finalized model/report changes to commit.')
else:
    subprocess.run(['git', '-C', str(repo), 'commit', '-m', 'model: record full YOLOX resistor comparison'], check=True)
    subprocess.run(git_auth + ['-C', str(repo), 'push', 'origin', f'HEAD:refs/heads/{BRANCH}'], check=True)
    print('Pushed finalized full result to', BRANCH)
del token